Licensed to the Apache Software Foundation (ASF) under one
or more contributor license agreements.  See the NOTICE file
distributed with this work for additional information
regarding copyright ownership.  The ASF licenses this file
to you under the Apache License, Version 2.0 (the
"License"); you may not use this file except in compliance
with the License.  You may obtain a copy of the License at

  http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing,
software distributed under the License is distributed on an
"AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY
KIND, either express or implied.  See the License for the
specific language governing permissions and limitations
under the License.

# Query Apache Pinot from Jupyter with JupySQL

This notebook queries a local Pinot **batch quickstart** from Jupyter using
[JupySQL](https://jupysql.ploomber.io/) SQL magics and [pinotdb](https://pypi.org/project/pinotdb/).

It covers:

1. Connecting to Pinot from a notebook
2. Running SQL (`SELECT`, `GROUP BY`, `ORDER BY`)
3. Plotting query results
4. Keeping results as a pandas DataFrame for later EDA or modeling

**Start Pinot first** (broker on port **8000**, controller on **9000**):

```bash
./build/bin/quick-start-batch.sh
```

or

```bash
docker run --name pinot-quickstart -p 2123:2123 -p 9000:9000 -p 8000:8000 -d apachepinot/pinot:latest QuickStart -type batch
```

The quickstart loads `baseballStats`. See `README.md` in this directory for install steps.

In [ ]:
from sqlalchemy import create_engine
import matplotlib.pyplot as plt

%matplotlib inline
%load_ext sql

%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

# Batch / Docker quickstart: broker 8000, controller 9000 (not 8099).
# Multi-stage is required for JupySQL %sqlplot, which rewrites plots as CTEs.
engine = create_engine(
    "pinot://localhost:8000/query/sql?controller=http://localhost:9000/",
    connect_args={"use_multistage_engine": "true"},
)
%sql engine

## Query Pinot with SQL magics

`%%sql` sends the statement to the Pinot broker (`POST /query/sql`).
Use `LIMIT` on exploratory scans. Aggregations on `baseballStats` are cheap.

In [ ]:
%%sql
SELECT playerName, teamID, yearID, runs, homeRuns
FROM baseballStats
LIMIT 5

In [ ]:
%%sql
SELECT playerName, SUM(runs) AS sum_runs
FROM baseballStats
WHERE yearID >= 2000
GROUP BY playerName
ORDER BY sum_runs DESC
LIMIT 10

## Plot query results

Assign a `%sql` result to a variable. With `SqlMagic.autopandas = True` you get a
DataFrame you can plot with matplotlib (or pass into `%sqlplot`).

In [ ]:
top_teams = %sql SELECT teamID, SUM(runs) AS total_runs FROM baseballStats GROUP BY teamID ORDER BY total_runs DESC LIMIT 10

ax = top_teams.plot.bar(x="teamID", y="total_runs", legend=False)
ax.set_title("Top 10 teams by total runs (baseballStats)")
ax.set_xlabel("teamID")
ax.set_ylabel("total runs")
plt.tight_layout()
plt.show()

In [ ]:
%%sql --save top_teams_sql
SELECT teamID, SUM(runs) AS total_runs
FROM baseballStats
GROUP BY teamID
ORDER BY total_runs DESC
LIMIT 10

In [ ]:
%sqlplot bar --table top_teams_sql --column teamID

## Keep results for EDA or modeling

The DataFrame is a normal pandas object. Use it for further EDA or as features
for a model — training a model is out of scope for this tutorial.

In [ ]:
player_runs = %sql SELECT playerName, SUM(runs) AS sum_runs, SUM(homeRuns) AS sum_hr FROM baseballStats WHERE yearID >= 2000 GROUP BY playerName ORDER BY sum_runs DESC LIMIT 20

print(player_runs.dtypes)
player_runs.head()